# Story 1.2: Simulation Graph E2E Test

This notebook tests the LangGraph simulation workflow end-to-end with real AI agents.

**What this tests:**
1. Complete scenario generation
2. LangGraph workflow with interrupt() for human-in-the-loop
3. Full simulation cycle through multiple turns
4. MLflow tracing of multi-step execution

## 1. Setup & Imports

In [1]:
import mlflow

from summit_sim.agents.generator import generate_scenario
from summit_sim.graphs.simulation import create_simulation_graph
from summit_sim.schemas import HostConfig
from summit_sim.settings import settings

# Initialize MLflow
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment(settings.mlflow_experiment_name)
mlflow.pydantic_ai.autolog()

print(f"✓ MLflow tracking URI: {settings.mlflow_tracking_uri}")
print(f"✓ Experiment name: {settings.mlflow_experiment_name}")
print(f"✓ Model: {settings.default_model}")
print(f"✓ API Key configured: {bool(settings.openrouter_api_key)}")

✓ MLflow tracking URI: https://mlflow.bhamm-lab.com
✓ Experiment name: summit-sim
✓ Model: google/gemini-3.1-flash-lite-preview
✓ API Key configured: True


## 2. Generate Scenario

First, we'll generate a complete scenario with multiple turns.

In [2]:
# Test configuration
config = HostConfig(num_participants=3, activity_type="hiking", difficulty="med")

print(f"Generating scenario: {config}")
print("-" * 60)

# Generate scenario
scenario = await generate_scenario(config)

print(f"✓ Title: {scenario.title}")
print(f"✓ Setting: {scenario.setting}")
print(f"✓ Patient: {scenario.patient_summary}")
print(f"✓ Turns: {len(scenario.turns)}")
print(f"✓ Starting turn: {scenario.starting_turn_id}")
print("\nLearning Objectives:")
for obj in scenario.learning_objectives:
    print(f"  • {obj}")

Generating scenario: num_participants=3 activity_type='hiking' difficulty='med'
------------------------------------------------------------
✓ Title: The Ridge Trail Incident
✓ Setting: High alpine ridgeline in the Cascade Mountains. Exposed, windy, dropping temperatures (late afternoon). 4 miles from the main trailhead. Terrain is technical and slow-going.
✓ Patient: 35-year-old male, experienced hiker. Complaining of severe right ankle pain following a fall. Currently cold to the touch, shivering, and pale. Right ankle shows deformity and swelling.
✓ Turns: 3
✓ Starting turn: 0

Learning Objectives:
  • Prioritize systemic patient stabilization (addressing hypothermia/shock) over localized injury treatment.
  • Perform a proper initial assessment (Scene Safety, Primary Survey, SAMPLE history).
  • Formulate an appropriate evacuation plan based on injury severity and environment.


Trace(trace_id=tr-bc00605f8f8b81d7218eee10e4492125)

## 3. Display Scenario Turns

Let's see what turns were generated:

In [3]:
for _i, turn in enumerate(scenario.turns):
    print()
    print("=" * 60)
    marker = "(START)" if turn.is_starting_turn else ""
    print(f"Turn {turn.turn_id} {marker}")
    print("=" * 60)
    print()
    print(turn.narrative_text)
    print()
    print("Choices:")
    for choice in turn.choices:
        if choice.next_turn_id is None:
            end = "END"
        else:
            end = f"Turn {choice.next_turn_id}"
        mark = "✓" if choice.is_correct else " "
        print(f"  [{mark}] {choice.description} -> {end}")


Turn 0 (START)

You are hiking with two friends on a remote ridgeline. One friend trips on a rock and falls hard on his right ankle. He is grimacing in pain and refuses to put weight on the limb. He is starting to shiver, and despite wearing a light fleece, he is turning pale. The wind is picking up, and temperatures are dropping as sunset approaches. You must decide on the next step.

Choices:
  [✓] Perform a rapid primary survey (C-spine stabilization, airway, breathing, circulation), then move the patient to a sheltered area to warm him up before evaluating the injury further. -> Turn 1
  [ ] Immediately remove the patient's boot to examine the ankle, and attempt to reduce the apparent deformity before performing a full assessment. -> Turn 1

Turn 1 

You successfully moved the patient to a sheltered spot behind a rock formation, added insulation layers, and assessed for other injuries. The patient's shivering has subsided slightly. The ankle remains deformed and swollen, but he is

## 4. Initialize Simulation Graph

Create the LangGraph workflow with checkpointer for interrupt support:

In [4]:
# Create the graph with in-memory checkpointer
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
graph = create_simulation_graph(checkpointer=memory)

# Initialize state (TypedDict - use dict syntax)
initial_state = {
    "scenario_draft": scenario,
    "current_turn_id": scenario.starting_turn_id,
    "transcript": [],
    "is_complete": False,
    "key_learning_moments": [],
    "last_selected_choice": None,
    "simulation_result": None,
}

# Config with thread_id required for checkpointer
config = {"configurable": {"thread_id": "notebook-thread"}}

print("✓ Graph created successfully")
print(f"✓ Initial state: turn_id={initial_state['current_turn_id']}")

✓ Graph created successfully
✓ Initial state: turn_id=0


## 5. Run Simulation

Execute the full simulation workflow. The graph will:
1. Present each turn
2. Pause with `interrupt()` for your choice
3. Call the AI agent for feedback
4. Advance to next turn
5. Repeat until complete

In [5]:
from langgraph.types import Command

current_state = initial_state
turn_count = 0
simulation_complete = False

print("\n=== STARTING SIMULATION ===\n")

while not simulation_complete:
    turn_count += 1
    print(f"\n--- TURN {turn_count} ---")

    async for event in graph.astream(current_state, config):
        if "__interrupt__" in event:
            interrupt_obj = event["__interrupt__"]
            if hasattr(interrupt_obj, "value"):
                interrupt_data = interrupt_obj.value
            elif isinstance(interrupt_obj, (list, tuple)):
                interrupt_data = (
                    interrupt_obj[0].value
                    if hasattr(interrupt_obj[0], "value")
                    else interrupt_obj[0]
                )
            else:
                interrupt_data = interrupt_obj

            print(f"\n📖 {interrupt_data['narrative']}\n")
            print("Choices:")
            for i, choice in enumerate(interrupt_data["choices"], 1):
                print(f"  {i}. {choice['description']}")

            while True:
                try:
                    sel = int(input("\nSelect: ")) - 1
                    if 0 <= sel < len(interrupt_data["choices"]):
                        break
                    print("Invalid")
                except ValueError:
                    print("Enter number")

            selected = interrupt_data["choices"][sel]
            current_state = await graph.ainvoke(
                Command(resume={"choice_id": selected["choice_id"]}), config
            )
            break

        # Check for graph completion
        if event == {}:
            simulation_complete = True
            break

    # Check if state indicates completion (AppState is TypedDict, use dict access)
    if current_state.get("is_complete"):
        simulation_complete = True

    # Safety limit
    if turn_count > 10:
        print("Safety stop")
        break

print("\n=== SIMULATION COMPLETE ===")


=== STARTING SIMULATION ===


--- TURN 1 ---

📖 You are hiking with two friends on a remote ridgeline. One friend trips on a rock and falls hard on his right ankle. He is grimacing in pain and refuses to put weight on the limb. He is starting to shiver, and despite wearing a light fleece, he is turning pale. The wind is picking up, and temperatures are dropping as sunset approaches. You must decide on the next step.

Choices:
  1. Perform a rapid primary survey (C-spine stabilization, airway, breathing, circulation), then move the patient to a sheltered area to warm him up before evaluating the injury further.
  2. Immediately remove the patient's boot to examine the ankle, and attempt to reduce the apparent deformity before performing a full assessment.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]



--- TURN 2 ---

📖 You successfully moved the patient to a sheltered spot behind a rock formation, added insulation layers, and assessed for other injuries. The patient's shivering has subsided slightly. The ankle remains deformed and swollen, but he is neurovascularly intact (distal pulses present, sensation intact). You are 4 miles from the trailhead over technical, rocky terrain, and it is two hours until dark.

Choices:
  1. Secure the ankle, insulate the patient, and prepare to shelter in place or slow-move while someone goes for help.
  2. Attempt an assisted 'crutch' walk to get the patient off the ridge and back to the car as fast as possible.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to sil


--- TURN 3 ---

📖 The patient is stable and has been given some pain relief according to protocols. The wind has intensified, and you are losing daylight rapidly. The terrain ahead is difficult. You have three people in your party total. How do you proceed?

Choices:
  1. Stay put, build a shelter, and signal for rescue using a satellite device or phone; keep the patient warm and comfortable until help arrives.
  2. Push through the pain and hike out immediately, believing the adrenaline will help him make it and that the injury isn't that bad.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]



=== SIMULATION COMPLETE ===


[Trace(trace_id=tr-db843573b85201b7cbf5b2228830d1c0), Trace(trace_id=tr-454eded9c93e3af066fcfd57faee63c1), Trace(trace_id=tr-c00248c002de812fd1c83f3567ae8b88)]

## 6. Display Results

Show the complete transcript and learning moments:

In [6]:
print("\n" + "=" * 60)
print("TRANSCRIPT")
print("=" * 60 + "\n")

for _i, entry in enumerate(current_state["transcript"], 1):
    print(f"Turn {entry['turn_id']}:")
    print(f"  Situation: {entry['turn_narrative'][:80]}...")
    print(f"  Choice: {entry['choice_description']}")
    print(f"  Feedback: {entry['feedback'][:100]}...")
    print(f"  Learning: {', '.join(entry['learning_moments'])}")
    print()


TRANSCRIPT

Turn 0:
  Situation: You are hiking with two friends on a remote ridgeline. One friend trips on a roc...
  Choice: Immediately remove the patient's boot to examine the ankle, and attempt to reduce the apparent deformity before performing a full assessment.
  Feedback: While it’s natural to want to address the visible pain, jumping straight to the ankle puts the patie...
  Learning: Always conduct a thorough primary survey before focusing on a specific injury., In cold, high-alpine environments, systemic stabilization—specifically preventing further heat loss—must take precedence over localized orthopaedic injury management.

Turn 0:
  Situation: You are hiking with two friends on a remote ridgeline. One friend trips on a roc...
  Choice: Immediately remove the patient's boot to examine the ankle, and attempt to reduce the apparent deformity before performing a full assessment.
  Feedback: While it’s natural to want to address the visible pain, jumping straight to the ankle

In [7]:
print("\n" + "=" * 60)
print("KEY LEARNING MOMENTS")
print("=" * 60 + "\n")

for _i, moment in enumerate(current_state["key_learning_moments"], 1):
    print(f"{_i}. {moment}")


KEY LEARNING MOMENTS

1. Always conduct a thorough primary survey before focusing on a specific injury.
2. In cold, high-alpine environments, systemic stabilization—specifically preventing further heat loss—must take precedence over localized orthopaedic injury management.
3. Always conduct a thorough primary survey before focusing on a specific injury.
4. In cold, high-alpine environments, systemic stabilization—specifically preventing further heat loss—must take precedence over localized orthopaedic injury management.
5. Always conduct a thorough primary survey before focusing on a specific injury.
6. In cold, high-alpine environments, systemic stabilization—specifically preventing further heat loss—must take precedence over localized orthopaedic injury management.
7. Always conduct a thorough primary survey before focusing on a specific injury.
8. In cold, high-alpine environments, systemic stabilization—specifically preventing further heat loss—must take precedence over localized 

## 7. MLflow Verification

Check your MLflow UI to verify:
- Multi-step execution traced under single Parent Run
- Each agent call logged with latency metrics
- Full conversation context preserved

In [8]:
# Show experiment info
experiment = mlflow.get_experiment_by_name(settings.mlflow_experiment_name)
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")
print(f"\nView traces at: {settings.mlflow_tracking_uri}")
print("\nLook for a trace showing:")
print("  - Scenario generation")
print("  - Multiple simulation turns")
print("  - All under a single parent run")

Experiment ID: 7
Artifact Location: mlflow-artifacts:/7

View traces at: https://mlflow.bhamm-lab.com

Look for a trace showing:
  - Scenario generation
  - Multiple simulation turns
  - All under a single parent run


---
## ✅ E2E Test Complete

If you successfully completed the simulation:
- ✓ LangGraph workflow executed correctly
- ✓ Human-in-the-loop interrupt worked
- ✓ AI feedback generated for each choice
- ✓ State advanced through all turns
- ✓ MLflow captured multi-step execution